In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
import math
import copy
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torch import tensor
from torch.nn.functional import pad

### Config

In [3]:
d_model=512
h=8
d_ff=2048
dropout=0.1
N=6

In [4]:
src_vocab, tgt_vocab = 11010, 19621

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Model

In [2]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

### Inference

In [6]:
position_encoding = PositionalEncoding(d_model, dropout)

In [7]:
src_embed = nn.Sequential(Embeddings(d_model, src_vocab), copy.deepcopy(position_encoding))

In [10]:
src_input = tensor([[
    0, 1101, 7165, 1913, 2433, 7559, 4354, 5914, 10814, 10805,
    10019, 4389, 9040, 2588, 10022, 20, 1, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2
]])

In [11]:
embed_result = src_embed(src_input)

In [12]:
embed_result.shape

torch.Size([1, 50, 512])

In [13]:
torch.save(embed_result, "embed_result.pt")